<a href="https://colab.research.google.com/github/mchadw/projSwing/blob/main/projSwing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

matches_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-matches.csv')
points_2020s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv')
points_2010s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2010s.csv')

points_raw = pd.concat([points_2010s_raw, points_2020s_raw])

matches_raw.head(10)
matches_raw.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_732/1140156045.py:8: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  points_2020s_raw = pd.read_csv('/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv')


(7566, 15)

In [ ]:
points_raw.head(10)

,match_id,Pt,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
0,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,1,0,0,0,0.0,0-0,1,True,1,4n,4b1b1b3n@,NaN,1
1,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,2,0,0,0,0.0,15-0,1,True,1,5n,5f28f2b1b1b2f3f3d@,NaN,1
2,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,3,0,0,0,0.0,30-0,1,True,1,4C,NaN,NaN,1
3,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,4,0,0,0,0.0,40-0,1,True,1,4r2d#,NaN,NaN,1
4,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,5,0,0,1,0.0,0-0,2,True,2,5d,c4b1b1d!@,NaN,1
5,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,6,0,0,1,0.0,0-15,2,True,2,4n,4f38f2b1b1b1b1w@,NaN,1
6,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,7,0,0,1,0.0,0-30,2,True,2,6f#,NaN,NaN,2
7,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,8,0,0,1,0.0,15-30,2,True,2,4f38f+3f1i^1i3n#,NaN,NaN,2
8,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,9,0,0,1,0.0,30-30,2,True,2,4w,6f17f3f3f3n#,NaN,1
9,20191124-M-Davis_Cup_Finals-F-Rafael_Nadal-Den...,10,0,0,1,0.0,30-40,2,True,2,4r17b1m3o3*,NaN,NaN,2


In [ ]:
import random
import re

PLAYER = "Carlos Alcaraz"

MATCHES_PATH = "/content/drive/MyDrive/projSwing/charting-m-matches.csv"
POINTS_2020S_PATH = "/content/drive/MyDrive/projSwing/charting-m-points-2020s.csv"
POINTS_2010S_PATH = "/content/drive/MyDrive/projSwing/charting-m-points-2010s.csv"

matches_raw = pd.read_csv(MATCHES_PATH)
points_2020s_raw = pd.read_csv(POINTS_2020S_PATH, low_memory=False)
points_2010s_raw = pd.read_csv(POINTS_2010S_PATH, low_memory=False)
points_raw = pd.concat([points_2010s_raw, points_2020s_raw], ignore_index=True)

p1_is_player = matches_raw["Player 1"].fillna("").str.contains(PLAYER, case=False, regex=False)
p2_is_player = matches_raw["Player 2"].fillna("").str.contains(PLAYER, case=False, regex=False)
# Keep matches where PLAYER is on exactly one side. Match date is retained because
# the evaluation design must reproduce the information ordering of future prediction.
keep = p1_is_player ^ p2_is_player
player_matches = matches_raw.loc[
    keep, ["match_id", "Date", "Player 1", "Player 2"]
].copy()
player_matches["player_side"] = 2
player_matches.loc[p1_is_player.loc[keep], "player_side"] = 1

match_meta = player_matches[["match_id", "Date", "player_side"]].drop_duplicates(
    "match_id"
)
player_points = points_raw.merge(match_meta, on="match_id", how="inner")

print(f"{PLAYER}: {match_meta['match_id'].nunique()} matches, {len(player_points)} points")

FH = set("frvhjulo")
BH = set("bszikymp")
SHOT = re.compile(r"([fbrsvzopuylmhijktq])[+\-=;^]*([0-3])?[789]?[nwdxe!g]*[*#@C]?")
SERVE_ONLY = re.compile(r"^c*[0456]?\+?[nwdxegV!]+$")
SERVE_PREFIX = re.compile(r"^c*[0456]?\+?")
LABELS = ["f1", "f2", "f3", "b1", "b2", "b3"]


def rally(first, second):
    """Return (shot buckets, rally_starts_with_return).

    A bucket is f/b plus direction 1-3, or None when the token is not one of those
    six categories. The serve number is not a bucket. The first bucket is the
    returner's shot. Depth codes 7-9 are intentionally omitted from this basic model.
    """
    code = first if isinstance(first, str) else ""
    if SERVE_ONLY.fullmatch(code):
        code = second if isinstance(second, str) else ""
    code = SERVE_PREFIX.sub("", code)
    starts_with_return = True
    shots = []
    for stroke, direction in SHOT.findall(code):
        hand = "f" if stroke in FH else "b" if stroke in BH else None
        shots.append(
            hand + direction
            if hand and direction in ("1", "2", "3")
            else None
        )
    return shots, starts_with_return


def hitters_for(n_shots, server, starts_with_return):
    """Player index (1 or 2) for each parsed shot, alternating from the first hitter."""
    server = int(server)
    returner = 3 - server
    hitter = returner if starts_with_return else server
    out = []
    for _ in range(n_shots):
        out.append(hitter)
        hitter = 3 - hitter
    return out


def response_detail(first, second, svr, player_side):
    """Return parsed shots and all opponent-to-player response opportunities.

    Candidate pairs preserve missing shot buckets. Eligible pairs are the subset for
    which both the incoming shot and the player's response belong to LABELS. Keeping
    these populations separate prevents model coverage from being confused with
    predictive accuracy.
    """
    shots, starts_with_return = rally(first, second)
    try:
        server = int(svr)
        side = int(player_side)
    except (TypeError, ValueError):
        return shots, [], [], []
    if server not in (1, 2) or side not in (1, 2):
        return shots, [], [], []

    who = hitters_for(len(shots), server, starts_with_return)
    opponent_side = 3 - side
    candidate_pairs = []
    for i in range(len(shots) - 1):
        # Eligibility is determined by the sequence of hitters, not by whether the
        # shot labels happen to be observable. This retains an auditable denominator.
        if who[i] == opponent_side and who[i + 1] == side:
            candidate_pairs.append((shots[i], shots[i + 1]))

    eligible_pairs = [
        (incoming, reply)
        for incoming, reply in candidate_pairs
        if incoming is not None and reply is not None
    ]
    return shots, who, candidate_pairs, eligible_pairs


def extract_response_data(points, n_examples=8, seed=0):
    """Create the analysis population and document losses caused by missing labels.

    The returned data frame has one row per eligible transition. It is the single
    source from which training and test tables should be drawn, ensuring that all
    models are compared on the same empirical observations.
    """
    rng = random.Random(seed)
    stats = {
        "points": 0,
        "empty_points": 0,
        "points_with_candidate": 0,
        "points_with_eligible": 0,
        "candidate_pairs": 0,
        "eligible_pairs": 0,
        "missing_incoming": 0,
        "missing_reply": 0,
        "missing_both": 0,
    }
    records = []
    examples = []

    columns = ["match_id", "Pt", "1st", "2nd", "Svr", "player_side"]
    for match_id, point_number, first, second, svr, side in points[columns].itertuples(
        index=False, name=None
    ):
        stats["points"] += 1
        shots, who, candidates, eligible = response_detail(first, second, svr, side)

        if not shots:
            stats["empty_points"] += 1
        if candidates:
            stats["points_with_candidate"] += 1
        if eligible:
            stats["points_with_eligible"] += 1

        for incoming, reply in candidates:
            stats["candidate_pairs"] += 1
            if incoming is None and reply is None:
                stats["missing_both"] += 1
            elif incoming is None:
                stats["missing_incoming"] += 1
            elif reply is None:
                stats["missing_reply"] += 1
            else:
                stats["eligible_pairs"] += 1

        for transition_number, (incoming, reply) in enumerate(eligible, start=1):
            records.append(
                {
                    "match_id": match_id,
                    "point_number": point_number,
                    "transition_number": transition_number,
                    "incoming": incoming,
                    "actual_response": reply,
                }
            )

        if eligible:
            example = {
                "match_id": match_id,
                "point_number": point_number,
                "1st": first,
                "2nd": second,
                "Svr": svr,
                "player_side": side,
                "shots": shots,
                "hitters": who,
                "candidates": [f"{a}->{b}" for a, b in candidates],
                "eligible": [f"{a}->{b}" for a, b in eligible],
            }
            # Reservoir sampling avoids privileging observations that occur early in
            # the source files while retaining a small, reproducible audit sample.
            if len(examples) < n_examples:
                examples.append(example)
            else:
                j = rng.randint(0, stats["points_with_eligible"] - 1)
                if j < n_examples:
                    examples[j] = example

    response_data = pd.DataFrame.from_records(
        records,
        columns=[
            "match_id",
            "point_number",
            "transition_number",
            "incoming",
            "actual_response",
        ],
    )
    return response_data, stats, examples


def build_response_matrix(response_data):
    """Count eligible incoming-shot -> player-response transitions."""
    matrix = pd.crosstab(
        response_data["incoming"], response_data["actual_response"], dropna=False
    ).reindex(index=LABELS, columns=LABELS, fill_value=0)
    matrix.index.name = "opponent_shot"
    matrix.columns.name = "player_response"
    return matrix


def make_evaluation_table(test_pairs, matrix_guesses, baseline_guesses):
    """Place both models' predictions beside the same held-out outcomes.

    Predictions are attached positionally only after verifying equal lengths. No
    observations are removed here: model-specific row filtering would make accuracy
    estimates incomparable. The policy for unavailable predictions is addressed
    separately when the models are fitted.
    """
    n_rows = len(test_pairs)
    if len(matrix_guesses) != n_rows or len(baseline_guesses) != n_rows:
        raise ValueError("Each model must provide one prediction per test transition")

    evaluation = test_pairs.reset_index(drop=True).copy()
    evaluation["matrix_guess"] = list(matrix_guesses)
    evaluation["baseline_guess"] = list(baseline_guesses)
    return evaluation


def _check_parser():
    shots, who, candidates, eligible = response_detail("4n", "4b1f2n@", 1, 1)
    assert shots == ["b1", "f2"], shots
    assert who == [2, 1], who
    assert candidates == [("b1", "f2")], candidates
    assert eligible == [("b1", "f2")], eligible

    shots, who, candidates, eligible = response_detail("4n", "4f1b2f3", 1, 2)
    assert shots == ["f1", "b2", "f3"], shots
    assert who == [2, 1, 2], who
    assert candidates == [("b2", "f3")], candidates
    assert eligible == [("b2", "f3")], eligible

    shots, who, candidates, eligible = response_detail("4C", None, 1, 1)
    assert shots == [] and candidates == [] and eligible == []

    shots, who, candidates, eligible = response_detail("6f1b2*", None, 1, 1)
    assert shots == ["f1", "b2"], shots
    assert who == [2, 1], who
    assert candidates == [("f1", "b2")], candidates
    assert eligible == [("f1", "b2")], eligible

    # A transition with an unclassified incoming shot remains in the candidate
    # population but is excluded from the six-category analysis population. This
    # distinction is required to calculate and report empirical coverage.
    shots, who, candidates, eligible = response_detail("6f1f0b2", None, 2, 1)
    assert shots[0] == "f1" and shots[1] is None and shots[2] == "b2"
    assert candidates == [(None, "b2")], candidates
    assert eligible == [], eligible


_check_parser()
print("parser, hitter, and eligibility checks passed")

response_data, stats, examples = extract_response_data(player_points)
# This full-data matrix is descriptive only. It summarizes the observed corpus but
# must not be used for prediction, because it includes the later test matches.
descriptive_matrix = build_response_matrix(response_data)

# Coverage is reported at the transition level because transition-level accuracy is
# the primary outcome. Point-level summaries remain useful descriptive diagnostics,
# but they do not share the denominator used to estimate predictive performance.
candidate_n = stats["candidate_pairs"]
eligible_n = stats["eligible_pairs"]
transition_coverage = eligible_n / candidate_n if candidate_n else float("nan")
point_n = stats["points"] or 1

print(f"matches represented in point data: {player_points['match_id'].nunique()}")
print(f"points: {stats['points']}")
print(f"candidate opponent -> {PLAYER} transitions: {candidate_n}")
print(f"eligible six-category transitions: {eligible_n}")
print(f"transition coverage: {transition_coverage:.1%}")
print("exclusions from the six-category analysis population:")
print(f"  missing incoming category: {stats['missing_incoming']}")
print(f"  missing response category: {stats['missing_reply']}")
print(f"  both categories missing: {stats['missing_both']}")
print(f"points with an empty shot list: {stats['empty_points'] / point_n:.1%}")
print(
    "points with at least one eligible transition: "
    f"{stats['points_with_eligible'] / point_n:.1%}"
)

# This assertion guards the link between the reported eligible population and the
# row-level data that will later be divided into training and test matches.
assert len(response_data) == eligible_n
print("\neligible transition table:")
print(response_data.head(10).to_string(index=False))

print("\ndescriptive opponent-shot counts (row sums):")
print(descriptive_matrix.sum(axis=1).astype(int))
low = descriptive_matrix.sum(axis=1)
print("rows with fewer than 30 opponent shots:", list(low[low < 30].index))
print(descriptive_matrix)

print("\nreproducible audit sample:")
print(pd.DataFrame(examples).to_string(index=False))

# A row of NaNs means that no eligible transition had that incoming bucket; it does
# not represent a uniform response distribution.
row_sums = descriptive_matrix.sum(axis=1)
prob_matrix = descriptive_matrix.div(row_sums.replace(0, pd.NA), axis=0)
print("\ndescriptive conditional response probabilities:")
print(prob_matrix)

Carlos Alcaraz: 222 matches, 36949 points
parser, hitter, and eligibility checks passed
matches represented in point data: 221
points: 36949
candidate opponent -> Carlos Alcaraz transitions: 51505
eligible six-category transitions: 50593
transition coverage: 98.2%
exclusions from the six-category analysis population:
  missing incoming category: 49
  missing response category: 672
  both categories missing: 191
points with an empty shot list: 13.5%
points with at least one eligible transition: 59.9%

eligible transition table:
                                              match_id  point_number  transition_number incoming actual_response
20190410-M-Murcia_CH-R32-Carlos_Alcaraz-Pedro_Martinez             2                  1       f3              b3
20190410-M-Murcia_CH-R32-Carlos_Alcaraz-Pedro_Martinez             3                  1       b1              f1
20190410-M-Murcia_CH-R32-Carlos_Alcaraz-Pedro_Martinez             4                  1       f3              b1
20190410-M-Murc

In [ ]:
def chronological_match_split(response_data, match_meta, train_fraction=0.80):
    """Partition complete matches in temporal order.

    The earliest matches constitute the training period and the latest matches form
    the test period. Splitting match identifiers, rather than transition rows,
    prevents observations from the same match or rally from entering both samples.
    """
    if not 0 < train_fraction < 1:
        raise ValueError("train_fraction must lie strictly between zero and one")

    # Restrict the split to matches represented in the eligible transition table.
    # Date parsing is explicit so malformed metadata cannot silently alter ordering.
    analysis_matches = (
        response_data[["match_id"]]
        .drop_duplicates()
        .merge(
            match_meta[["match_id", "Date"]],
            on="match_id",
            how="left",
            validate="one_to_one",
        )
    )
    date_text = analysis_matches["Date"].astype("string").str.replace(
        r"\.0$", "", regex=True
    )
    analysis_matches["match_date"] = pd.to_datetime(
        date_text, format="%Y%m%d", errors="coerce"
    )
    invalid_dates = analysis_matches.loc[
        analysis_matches["match_date"].isna(), "match_id"
    ].tolist()
    if invalid_dates:
        raise ValueError(f"Missing or malformed dates for matches: {invalid_dates}")

    analysis_matches = analysis_matches.sort_values(
        ["match_date", "match_id"]
    ).reset_index(drop=True)
    n_matches = len(analysis_matches)
    target_n_train = int(n_matches * train_fraction)
    if target_n_train == 0 or target_n_train == n_matches:
        raise ValueError("The requested split leaves one partition without matches")

    # The first provisional test date defines the historical cutoff. Every match on
    # that date is assigned to testing, rather than ordering same-day matches by an
    # arbitrary identifier. The realized fraction can therefore differ slightly
    # from 80%, but the temporal interpretation remains defensible.
    cutoff_date = analysis_matches.loc[target_n_train, "match_date"]
    analysis_matches["split"] = "test"
    analysis_matches.loc[
        analysis_matches["match_date"] < cutoff_date, "split"
    ] = "train"
    if (analysis_matches["split"] == "train").sum() == 0:
        raise ValueError("The date-aware cutoff leaves no matches for training")

    train_ids = set(
        analysis_matches.loc[analysis_matches["split"] == "train", "match_id"]
    )
    test_ids = set(
        analysis_matches.loc[analysis_matches["split"] == "test", "match_id"]
    )

    train_pairs = response_data[response_data["match_id"].isin(train_ids)].copy()
    test_pairs = response_data[response_data["match_id"].isin(test_ids)].copy()

    # These assertions encode the study design: complete coverage, no overlap, and
    # strict temporal separation between the training and test periods.
    observed_ids = set(response_data["match_id"])
    assert train_ids.isdisjoint(test_ids)
    assert train_ids | test_ids == observed_ids
    assert train_pairs["match_id"].nunique() == len(train_ids)
    assert test_pairs["match_id"].nunique() == len(test_ids)
    assert (
        analysis_matches.loc[analysis_matches["split"] == "train", "match_date"].max()
        < analysis_matches.loc[
            analysis_matches["split"] == "test", "match_date"
        ].min()
    )
    return train_pairs, test_pairs, analysis_matches


def modal_response(counts):
    """Select the most frequent response using a pre-specified tie rule."""
    maximum = counts.max()
    if maximum <= 0:
        return None
    # LABELS defines a deterministic priority order. Fixing this rule before test
    # evaluation prevents arbitrary or outcome-informed resolution of exact ties.
    return next(label for label in LABELS if counts[label] == maximum)


# Change 1: create a historical 80/20 split at the match level. The test period is
# not consulted while either prediction rule is estimated.
train_pairs, test_pairs, match_split = chronological_match_split(
    response_data, match_meta, train_fraction=0.80
)
train_match_dates = match_split.loc[match_split["split"] == "train", "match_date"]
test_match_dates = match_split.loc[match_split["split"] == "test", "match_date"]

print("chronological evaluation design:")
print(
    f"  training matches: {train_pairs['match_id'].nunique()} "
    f"({train_match_dates.min().date()} through {train_match_dates.max().date()})"
)
print(
    f"  test matches: {test_pairs['match_id'].nunique()} "
    f"({test_match_dates.min().date()} through {test_match_dates.max().date()})"
)
print(f"  training transitions: {len(train_pairs)}")
print(f"  test transitions: {len(test_pairs)}")
print("\nmatches surrounding the temporal cutoff:")
cutoff_index = (match_split["split"] == "train").sum()
print(match_split.iloc[cutoff_index - 3 : cutoff_index + 3].to_string(index=False))

# Change 2: estimate both prediction rules exclusively from training transitions.
# The transition model conditions on the incoming category; the baseline ignores it
# and always predicts the globally modal response in the same training population.
train_matrix = build_response_matrix(train_pairs)

row_sums = train_matrix.sum(axis=1)
train_probs = train_matrix.div(row_sums.replace(0, np.nan), axis=0)

print("training transition probabilities:")
print(train_probs.round(3))
print(train_probs.sum(axis=1))  # each row should be 1.0 (NaN for empty rows)

matrix_rule = {
    incoming: modal_response(train_matrix.loc[incoming]) for incoming in LABELS
}
baseline_counts = (
    train_pairs["actual_response"].value_counts().reindex(LABELS, fill_value=0)
)
baseline_response = modal_response(baseline_counts)

print("\ntraining transition matrix:")
print(train_matrix)
print("matrix prediction rule:", matrix_rule)
print("baseline response:", baseline_response)

# Change 3: generate predictions only after fitting is complete, then place both
# guessers beside the same untouched outcomes. No model-specific filtering occurs.
matrix_guesses = test_pairs["incoming"].map(matrix_rule)
baseline_guesses = pd.Series(
    baseline_response, index=test_pairs.index, dtype="object"
)
evaluation = make_evaluation_table(
    test_pairs, matrix_guesses, baseline_guesses
)

unavailable = int(evaluation["matrix_guess"].isna().sum())
if unavailable:
    raise ValueError(
        f"The transition model lacks a training rule for {unavailable} test rows; "
        "an unseen-category policy is required before accuracy can be estimated."
    )

evaluation["matrix_correct"] = (
    evaluation["matrix_guess"] == evaluation["actual_response"]
)
evaluation["baseline_correct"] = (
    evaluation["baseline_guess"] == evaluation["actual_response"]
)
matrix_accuracy = evaluation["matrix_correct"].mean()
baseline_accuracy = evaluation["baseline_correct"].mean()

print("\nheld-out transition-level results:")
print(f"  transition-matrix accuracy: {matrix_accuracy:.1%}")
print(f"  modal-response baseline accuracy: {baseline_accuracy:.1%}")
print(f"  absolute accuracy difference: {matrix_accuracy - baseline_accuracy:+.1%}")
print("\nshared evaluation table:")
print(evaluation.head(10).to_string(index=False))

chronological evaluation design:
  training matches: 176 (2019-04-02 through 2025-05-28)
  test matches: 44 (2025-06-01 through 2026-04-12)
  training transitions: 41584
  test transitions: 9009

matches surrounding the temporal cutoff:
                                                    match_id     Date match_date split
      20250518-M-Rome_Masters-F-Jannik_Sinner-Carlos_Alcaraz 20250518 2025-05-18 train
20250526-M-Roland_Garros-R128-Carlos_Alcaraz-Giulio_Zeppieri 20250526 2025-05-26 train
 20250528-M-Roland_Garros-R64-Fabian_Marozsan-Carlos_Alcaraz 20250528 2025-05-28 train
     20250601-M-Roland_Garros-R16-Ben_Shelton-Carlos_Alcaraz 20250601 2025-06-01  test
       20250603-M-Roland_Garros-QF-Carlos_Alcaraz-Tommy_Paul 20250603 2025-06-03  test
  20250606-M-Roland_Garros-SF-Lorenzo_Musetti-Carlos_Alcaraz 20250606 2025-06-06  test
training transition probabilities:
player_response     f1     f2     f3     b1     b2     b3
opponent_shot                                            
f1 

In [ ]:
MIN_TEST_COUNT = 30


def rank_training_responses(train_matrix, labels=LABELS):
    """Lock each incoming shot's modal and runner-up replies from training only.

    Exact ties are broken by the pre-specified priority order in `labels`,
    matching `modal_response`.
    """
    rows = []
    for incoming in labels:
        counts = (
            train_matrix.loc[incoming].reindex(labels, fill_value=0)
            if incoming in train_matrix.index
            else pd.Series(0, index=labels)
        )
        row_total = int(counts.sum())
        ranked = sorted(labels, key=lambda lbl: (-int(counts[lbl]), labels.index(lbl)))
        locked_guess = ranked[0] if counts[ranked[0]] > 0 else None
        runner_up = (
            ranked[1] if len(ranked) > 1 and counts[ranked[1]] > 0 else None
        )
        train_p1 = (
            float(counts[locked_guess]) / row_total
            if locked_guess is not None and row_total > 0
            else np.nan
        )
        train_p2 = (
            float(counts[runner_up]) / row_total
            if runner_up is not None and row_total > 0
            else np.nan
        )
        rows.append(
            {
                "incoming": incoming,
                "train_n": row_total,
                "locked_guess": locked_guess,
                "runner_up": runner_up,
                "locked_wing": locked_guess[0] if locked_guess is not None else None,
                "train_p1": train_p1,
                "train_p2": train_p2,
                "train_margin": (
                    train_p1 - train_p2 if runner_up is not None else np.nan
                ),
            }
        )

    return pd.DataFrame(rows).set_index("incoming")


def _safe_ci(samples, alpha):
    """Return (low, high) percentile bounds, ignoring draws with zero denominator."""
    finite = samples[np.isfinite(samples)]
    if finite.size == 0:
        return np.nan, np.nan
    low, high = np.percentile(finite, [100 * (alpha / 2), 100 * (1 - alpha / 2)])
    return float(low), float(high)


def bootstrap_match_predictability(
    eval_df, locked_rules, labels=LABELS, n_boot=2000, ci=0.95, seed=0
):
    """Cluster-bootstrap held-out predictability metrics by resampling matches.

    Resampling complete test matches preserves within-rally and within-match
    dependence. Pre-aggregating counts per (match_id, incoming) produces the
    exact same transition-weighted ratios as concatenating resampled matches.
    """
    alpha = 1.0 - ci
    matches = eval_df["match_id"].drop_duplicates().to_numpy()
    n_matches = len(matches)
    k = len(labels)

    empty_cols = [
        "hit_ci_low",
        "hit_ci_high",
        "margin_ci_low",
        "margin_ci_high",
        "top2_ci_low",
        "top2_ci_high",
    ]
    if n_matches == 0:
        return (
            pd.DataFrame(np.nan, index=pd.Index(labels, name="incoming"), columns=empty_cols),
            (np.nan, np.nan),
        )

    match_to_idx = {m: idx for idx, m in enumerate(matches)}
    label_to_idx = {lbl: idx for idx, lbl in enumerate(labels)}

    m_idx = eval_df["match_id"].map(match_to_idx).to_numpy()
    in_idx = eval_df["incoming"].map(label_to_idx).to_numpy()
    act = eval_df["actual_response"]

    guess_series = eval_df["incoming"].map(locked_rules["locked_guess"])
    runner_series = eval_df["incoming"].map(locked_rules["runner_up"])

    is_p1 = (act == guess_series).to_numpy(dtype=np.int64)
    is_p2 = (act == runner_series).to_numpy(dtype=np.int64)

    n_mat = np.zeros((n_matches, k), dtype=np.float64)
    p1_mat = np.zeros((n_matches, k), dtype=np.float64)
    p2_mat = np.zeros((n_matches, k), dtype=np.float64)

    np.add.at(n_mat, (m_idx, in_idx), 1)
    np.add.at(p1_mat, (m_idx, in_idx), is_p1)
    np.add.at(p2_mat, (m_idx, in_idx), is_p2)

    rng = np.random.default_rng(seed)
    weights = rng.multinomial(
        n_matches, np.full(n_matches, 1.0 / n_matches), size=n_boot
    ).astype(np.float64)

    boot_n = weights @ n_mat
    boot_p1_hits = weights @ p1_mat
    boot_p2_hits = weights @ p2_mat

    with np.errstate(divide="ignore", invalid="ignore"):
        boot_p1 = np.where(boot_n > 0, boot_p1_hits / boot_n, np.nan)
        boot_p2 = np.where(boot_n > 0, boot_p2_hits / boot_n, np.nan)
        boot_margin = boot_p1 - boot_p2
        boot_top2 = boot_p1 + boot_p2
        boot_overall_n = boot_n.sum(axis=1)
        boot_overall_p1 = np.where(
            boot_overall_n > 0, boot_p1_hits.sum(axis=1) / boot_overall_n, np.nan
        )

    ci_rows = []
    for j, incoming in enumerate(labels):
        has_guess = pd.notna(locked_rules.loc[incoming, "locked_guess"])
        has_runner = pd.notna(locked_rules.loc[incoming, "runner_up"])
        hit_lo, hit_hi = _safe_ci(boot_p1[:, j], alpha) if has_guess else (np.nan, np.nan)
        if has_guess and has_runner:
            m_lo, m_hi = _safe_ci(boot_margin[:, j], alpha)
            t2_lo, t2_hi = _safe_ci(boot_top2[:, j], alpha)
        else:
            m_lo, m_hi = np.nan, np.nan
            t2_lo, t2_hi = np.nan, np.nan
        ci_rows.append(
            {
                "incoming": incoming,
                "hit_ci_low": hit_lo,
                "hit_ci_high": hit_hi,
                "margin_ci_low": m_lo,
                "margin_ci_high": m_hi,
                "top2_ci_low": t2_lo,
                "top2_ci_high": t2_hi,
            }
        )

    overall_ci = _safe_ci(boot_overall_p1, alpha)
    return pd.DataFrame(ci_rows).set_index("incoming"), overall_ci


def shuffle_null_predictability(
    eval_df, locked_rules, observed_hits, row_counts, labels=LABELS, n_perm=1000, seed=1
):
    """Estimate per-row hit rates when incoming labels are decoupled from replies.

    Shuffling `incoming` across held-out transitions destroys the conditional
    incoming -> reply link while preserving the marginal reply distribution.
    """
    k = len(labels)
    label_to_idx = {lbl: idx for idx, lbl in enumerate(labels)}

    in_idx = eval_df["incoming"].map(label_to_idx).to_numpy(dtype=np.int64)
    act_idx = eval_df["actual_response"].map(label_to_idx).to_numpy(dtype=np.int64)
    guess_idx = np.array(
        [
            label_to_idx[locked_rules.loc[lbl, "locked_guess"]]
            if pd.notna(locked_rules.loc[lbl, "locked_guess"])
            else -1
            for lbl in labels
        ],
        dtype=np.int64,
    )

    rng = np.random.default_rng(seed)
    null_hits = np.zeros((n_perm, k), dtype=np.int64)
    for b in range(n_perm):
        shuffled_in = rng.permutation(in_idx)
        hits = act_idx == guess_idx[shuffled_in]
        null_hits[b] = np.bincount(shuffled_in, weights=hits, minlength=k).astype(np.int64)

    rows = []
    for j, incoming in enumerate(labels):
        n_j = int(row_counts[j])
        if n_j > 0 and guess_idx[j] >= 0:
            null_rate = float(null_hits[:, j].mean()) / n_j
            obs_rate = float(observed_hits[j]) / n_j
            lift = obs_rate - null_rate
            # Add-one smoothing avoids zero p-values under finite permutations.
            null_p = (int((null_hits[:, j] >= observed_hits[j]).sum()) + 1) / (n_perm + 1)
        else:
            null_rate, lift, null_p = np.nan, np.nan, np.nan
        rows.append(
            {
                "incoming": incoming,
                "null_rate": null_rate,
                "lift_vs_null": lift,
                "null_p": null_p,
            }
        )

    return pd.DataFrame(rows).set_index("incoming")


def compute_predictability_report(
    train_matrix,
    eval_df,
    labels=LABELS,
    min_count=MIN_TEST_COUNT,
    n_boot=2000,
    n_perm=1000,
    seed=0,
):
    """Compute per-incoming-shot held-out predictability metrics and diagnostics.

    All decision rules (modal guess, runner-up, locked wing) are frozen from
    `train_matrix` and scored strictly on `eval_df`.
    """
    if not eval_df["incoming"].isin(labels).all() or not eval_df["actual_response"].isin(labels).all():
        raise ValueError("eval_df contains shot categories outside `labels`")

    locked_rules = rank_training_responses(train_matrix, labels=labels)

    expected_guess = eval_df["incoming"].map(locked_rules["locked_guess"])
    if "matrix_guess" in eval_df.columns:
        if not eval_df["matrix_guess"].fillna("").eq(expected_guess.fillna("")).all():
            raise ValueError(
                "eval_df['matrix_guess'] disagrees with the training-locked modal rule"
            )

    row_records = []
    observed_hits = np.zeros(len(labels), dtype=np.int64)
    row_counts = np.zeros(len(labels), dtype=np.int64)

    for j, incoming in enumerate(labels):
        rule = locked_rules.loc[incoming]
        locked_guess = rule["locked_guess"]
        runner_up = rule["runner_up"]
        locked_wing = rule["locked_wing"]

        sub = eval_df[eval_df["incoming"] == incoming]
        test_n = len(sub)
        row_counts[j] = test_n
        test_matches = int(sub["match_id"].nunique()) if test_n > 0 else 0

        if test_n > 0 and pd.notna(locked_guess):
            p1_hits = int((sub["actual_response"] == locked_guess).sum())
            observed_hits[j] = p1_hits
            hit_rate = p1_hits / test_n

            if pd.notna(runner_up):
                p2_hits = int((sub["actual_response"] == runner_up).sum())
                runner_up_rate = p2_hits / test_n
                margin = hit_rate - runner_up_rate
                top2_rate = hit_rate + runner_up_rate
            else:
                runner_up_rate = np.nan
                margin = np.nan
                top2_rate = np.nan

            wing_hits = int((sub["actual_response"].str[0] == locked_wing).sum())
            wing_rate = wing_hits / test_n
            dir_given_wing = p1_hits / wing_hits if wing_hits > 0 else np.nan

            drift = hit_rate - rule["train_p1"]
            test_counts = sub["actual_response"].value_counts().reindex(labels, fill_value=0)
            test_mode = modal_response(test_counts)
            mode_shifted = test_mode != locked_guess
        else:
            hit_rate = np.nan
            runner_up_rate = np.nan
            margin = np.nan
            top2_rate = np.nan
            wing_rate = np.nan
            dir_given_wing = np.nan
            drift = np.nan
            test_mode = None
            mode_shifted = False

        row_records.append(
            {
                "incoming": incoming,
                "test_n": test_n,
                "test_matches": test_matches,
                "low_n": test_n < min_count,
                "locked_guess": locked_guess,
                "hit_rate": hit_rate,
                "runner_up": runner_up,
                "runner_up_rate": runner_up_rate,
                "margin": margin,
                "top2_rate": top2_rate,
                "locked_wing": locked_wing,
                "wing_rate": wing_rate,
                "dir_given_wing": dir_given_wing,
                "train_p1": rule["train_p1"],
                "drift": drift,
                "test_mode": test_mode,
                "mode_shifted": mode_shifted,
            }
        )

    report = pd.DataFrame(row_records).set_index("incoming")

    boot_cis, overall_ci = bootstrap_match_predictability(
        eval_df, locked_rules, labels=labels, n_boot=n_boot, seed=seed
    )
    null_stats = shuffle_null_predictability(
        eval_df,
        locked_rules,
        observed_hits=observed_hits,
        row_counts=row_counts,
        labels=labels,
        n_perm=n_perm,
        seed=seed + 1,
    )
    report = report.join(boot_cis).join(null_stats)

    total_n = int(row_counts.sum())
    overall_hit_rate = (
        float(observed_hits.sum()) / total_n if total_n > 0 else np.nan
    )

    # Guard core algebraic invariants: row rates recombine to overall accuracy,
    # and hit_rate == wing_rate * dir_given_wing wherever wing_rate > 0.
    valid_rows = report["hit_rate"].notna()
    if total_n > 0 and valid_rows.all():
        weighted_hit = (
            (report["hit_rate"] * report["test_n"]).sum() / report["test_n"].sum()
        )
        assert np.isclose(weighted_hit, overall_hit_rate)
    wing_pos = report["wing_rate"].fillna(0) > 0
    if wing_pos.any():
        assert np.allclose(
            report.loc[wing_pos, "wing_rate"] * report.loc[wing_pos, "dir_given_wing"],
            report.loc[wing_pos, "hit_rate"],
        )

    overall = {
        "test_n": total_n,
        "test_matches": int(eval_df["match_id"].nunique()),
        "hit_rate": overall_hit_rate,
        "hit_ci_low": overall_ci[0],
        "hit_ci_high": overall_ci[1],
    }
    return report, overall


def _fmt_pct(val, signed=False):
    if pd.isna(val):
        return "—"
    return f"{val:+.1%}" if signed else f"{val:.1%}"


def _fmt_ci(lo, hi, signed=False):
    if pd.isna(lo) or pd.isna(hi):
        return "—"
    return f"[{_fmt_pct(lo, signed)}, {_fmt_pct(hi, signed)}]"


def format_predictability_tables(report):
    """Split the full report into two concise coach-readable display tables."""
    scouting = pd.DataFrame(
        {
            "test_n": report["test_n"],
            "matches": report["test_matches"],
            "low_n": report["low_n"],
            "guess": report["locked_guess"],
            "hit_rate": report["hit_rate"].map(_fmt_pct),
            "hit_95ci": [
                _fmt_ci(lo, hi)
                for lo, hi in zip(report["hit_ci_low"], report["hit_ci_high"])
            ],
            "runner_up": report["runner_up"],
            "margin": report["margin"].map(lambda v: _fmt_pct(v, signed=True)),
            "margin_95ci": [
                _fmt_ci(lo, hi, signed=True)
                for lo, hi in zip(report["margin_ci_low"], report["margin_ci_high"])
            ],
            "top2_rate": report["top2_rate"].map(_fmt_pct),
        },
        index=report.index,
    )
    diagnostics = pd.DataFrame(
        {
            "guess": report["locked_guess"],
            "wing_rate": report["wing_rate"].map(_fmt_pct),
            "dir|wing": report["dir_given_wing"].map(_fmt_pct),
            "train_p1": report["train_p1"].map(_fmt_pct),
            "drift": report["drift"].map(lambda v: _fmt_pct(v, signed=True)),
            "test_mode": report["test_mode"],
            "shifted": report["mode_shifted"],
            "null_rate": report["null_rate"].map(_fmt_pct),
            "lift": report["lift_vs_null"].map(lambda v: _fmt_pct(v, signed=True)),
            "null_p": report["null_p"].map(
                lambda v: "—" if pd.isna(v) else f"{v:.3f}"
            ),
        },
        index=report.index,
    )
    return scouting, diagnostics


def _check_predictability():
    # Synthetic training matrix with an intentional exact tie on f2 (f1 == f3 == 5)
    # to verify tie-breaking matches LABELS priority (f1 before f3).
    synth_train = pd.DataFrame(0, index=LABELS, columns=LABELS)
    synth_train.loc["f1", ["f1", "f3", "b1"]] = [6, 3, 1]
    synth_train.loc["f2", ["f1", "f3"]] = [5, 5]
    synth_train.loc["f3", ["b3", "b2"]] = [8, 2]
    synth_train.loc["b1", ["f1", "f2"]] = [7, 3]
    synth_train.loc["b2", ["f1", "f3"]] = [6, 4]
    synth_train.loc["b3", ["b3", "b1"]] = [9, 1]

    rules = rank_training_responses(synth_train)
    assert rules.loc["f2", "locked_guess"] == "f1"
    assert rules.loc["f2", "runner_up"] == "f3"
    for lbl in LABELS:
        assert rules.loc[lbl, "locked_guess"] == modal_response(synth_train.loc[lbl])

    # Two test matches with opposite outcomes on f1 (100% vs 0% hit rate) and an
    # intentionally absent incoming category (b1) to test empty-row handling.
    synth_eval = pd.DataFrame(
        [
            {"match_id": "m1", "incoming": "f1", "actual_response": "f1"},
            {"match_id": "m1", "incoming": "f1", "actual_response": "f1"},
            {"match_id": "m2", "incoming": "f1", "actual_response": "f3"},
            {"match_id": "m2", "incoming": "f1", "actual_response": "b1"},
            {"match_id": "m1", "incoming": "f3", "actual_response": "b2"},
            {"match_id": "m2", "incoming": "f3", "actual_response": "b2"},
        ]
    )
    rep, overall = compute_predictability_report(
        synth_train, synth_eval, min_count=3, n_boot=400, n_perm=200, seed=0
    )

    # f1: 2 hits (f1), 1 runner-up (f3), 3 forehands out of 4 transitions.
    assert np.isclose(rep.loc["f1", "hit_rate"], 0.5)
    assert np.isclose(rep.loc["f1", "runner_up_rate"], 0.25)
    assert np.isclose(rep.loc["f1", "margin"], 0.25)
    assert np.isclose(rep.loc["f1", "top2_rate"], 0.75)
    assert np.isclose(rep.loc["f1", "wing_rate"], 0.75)
    assert np.isclose(rep.loc["f1", "dir_given_wing"], 2 / 3)
    assert np.isclose(rep.loc["f1", "drift"], 0.5 - 0.6)
    assert rep.loc["f1", "hit_ci_low"] == 0.0 and rep.loc["f1", "hit_ci_high"] == 1.0

    # f3: training mode was b3, test hits are all b2 -> mode_shifted must flag it.
    assert rep.loc["f3", "test_mode"] == "b2"
    assert bool(rep.loc["f3", "mode_shifted"]) is True

    # b1 had zero test rows: must stay NaN and be flagged low_n.
    assert rep.loc["b1", "test_n"] == 0
    assert bool(rep.loc["b1", "low_n"]) is True
    assert pd.isna(rep.loc["b1", "hit_rate"])
    assert np.isclose(overall["hit_rate"], 2 / 6)


_check_predictability()
print("predictability metric checks passed")

predictability_report, predictability_overall = compute_predictability_report(
    train_matrix,
    evaluation,
    labels=LABELS,
    min_count=MIN_TEST_COUNT,
    n_boot=2000,
    n_perm=1000,
    seed=0,
)

assert np.isclose(predictability_overall["hit_rate"], matrix_accuracy)

print(
    f"\noverall held-out predictability ({predictability_overall['test_matches']} matches, "
    f"{predictability_overall['test_n']} transitions): "
    f"{_fmt_pct(predictability_overall['hit_rate'])} "
    f"95% match-bootstrap CI "
    f"{_fmt_ci(predictability_overall['hit_ci_low'], predictability_overall['hit_ci_high'])}"
)

scouting_table, diagnostic_table = format_predictability_tables(predictability_report)
print("\nscouting predictability by incoming shot (locked on training, scored on test):")
print(scouting_table.to_string())
print("\nwing/direction decomposition, calibration drift, and shuffle null check:")
print(diagnostic_table.to_string())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

SHOT_NAMES = {
    "f1": "FH Deuce",
    "f2": "FH Mid",
    "f3": "FH Ad",
    "b1": "BH Deuce",
    "b2": "BH Mid",
    "b3": "BH Ad",
}

PALETTE = {
    "fh_primary": "#1D4E89",
    "fh_secondary": "#5B8DBE",
    "fh_muted": "#C7D9EC",
    "bh_primary": "#B83B2A",
    "bh_secondary": "#DF7E6E",
    "bh_muted": "#F3CFC9",
    "break_win": "#1B7F5C",
    "habit_win": "#1D4E89",
    "neutral": "#6B7280",
    "slate": "#1F2937",
    "grid": "#E5E7EB",
}


def attach_point_outcomes(eval_df, points_df):
    """Attach point-level win/loss outcomes onto held-out transition rows.

    Deduplicates point records safely and drops any point with missing or
    conflicting PtWinner values instead of silently choosing one.
    """
    outcomes = points_df[["match_id", "Pt", "PtWinner", "player_side"]].copy()
    outcomes["PtWinner"] = pd.to_numeric(outcomes["PtWinner"], errors="coerce")
    outcomes["player_side"] = pd.to_numeric(outcomes["player_side"], errors="coerce")
    valid = outcomes["PtWinner"].isin([1, 2]) & outcomes["player_side"].isin([1, 2])
    outcomes = outcomes.loc[valid].copy()
    outcomes["player_won"] = outcomes["PtWinner"].eq(outcomes["player_side"])

    # Collapse duplicate (match_id, Pt) rows only when they agree on the winner.
    grouped = outcomes.groupby(["match_id", "Pt"], as_index=False)["player_won"].agg(
        n_unique="nunique", player_won="first"
    )
    consistent = grouped.loc[grouped["n_unique"] == 1, ["match_id", "Pt", "player_won"]]

    merged = eval_df.merge(
        consistent.rename(columns={"Pt": "point_number"}),
        on=["match_id", "point_number"],
        how="left",
        validate="many_to_one",
    )
    return merged


def compute_habit_payoff(
    eval_with_outcomes,
    locked_rules,
    labels=LABELS,
    n_boot=2000,
    ci=0.95,
    seed=0,
):
    """Compare held-out point-win rates for locked habits vs. pattern breaks.

    Uses match-cluster bootstrap CIs so transitions sharing a rally or match
    are resampled together rather than treated as independent points.
    """
    valid = eval_with_outcomes.loc[eval_with_outcomes["player_won"].notna()].copy()
    coverage = (
        len(valid) / len(eval_with_outcomes) if len(eval_with_outcomes) > 0 else np.nan
    )
    alpha = 1.0 - ci

    k = len(labels)
    matches = valid["match_id"].drop_duplicates().to_numpy()
    n_matches = len(matches)
    label_to_idx = {lbl: idx for idx, lbl in enumerate(labels)}

    rows = []
    if n_matches > 0:
        match_to_idx = {m: idx for idx, m in enumerate(matches)}
        m_idx = valid["match_id"].map(match_to_idx).to_numpy()
        in_idx = valid["incoming"].map(label_to_idx).to_numpy()
        guess = valid["incoming"].map(locked_rules["locked_guess"])
        is_habit = (valid["actual_response"] == guess).to_numpy(dtype=np.int64)
        is_break = 1 - is_habit
        won = valid["player_won"].to_numpy(dtype=np.int64)

        h_n = np.zeros((n_matches, k), dtype=np.float64)
        h_w = np.zeros((n_matches, k), dtype=np.float64)
        b_n = np.zeros((n_matches, k), dtype=np.float64)
        b_w = np.zeros((n_matches, k), dtype=np.float64)

        np.add.at(h_n, (m_idx, in_idx), is_habit)
        np.add.at(h_w, (m_idx, in_idx), is_habit * won)
        np.add.at(b_n, (m_idx, in_idx), is_break)
        np.add.at(b_w, (m_idx, in_idx), is_break * won)

        rng = np.random.default_rng(seed)
        weights = rng.multinomial(
            n_matches, np.full(n_matches, 1.0 / n_matches), size=n_boot
        ).astype(np.float64)

        boot_hn = weights @ h_n
        boot_hw = weights @ h_w
        boot_bn = weights @ b_n
        boot_bw = weights @ b_w

        with np.errstate(divide="ignore", invalid="ignore"):
            boot_h_rate = np.where(boot_hn > 0, boot_hw / boot_hn, np.nan)
            boot_b_rate = np.where(boot_bn > 0, boot_bw / boot_bn, np.nan)
            boot_diff = boot_b_rate - boot_h_rate

        tot_hn = h_n.sum(axis=0)
        tot_hw = h_w.sum(axis=0)
        tot_bn = b_n.sum(axis=0)
        tot_bw = b_w.sum(axis=0)
    else:
        tot_hn = tot_hw = tot_bn = tot_bw = np.zeros(k)
        boot_diff = np.full((1, k), np.nan)

    for j, incoming in enumerate(labels):
        has_guess = pd.notna(locked_rules.loc[incoming, "locked_guess"])
        hn = int(tot_hn[j]) if has_guess else 0
        bn = int(tot_bn[j]) if has_guess else 0
        h_rate = float(tot_hw[j]) / hn if hn > 0 else np.nan
        b_rate = float(tot_bw[j]) / bn if bn > 0 else np.nan
        diff = b_rate - h_rate if pd.notna(h_rate) and pd.notna(b_rate) else np.nan
        diff_lo, diff_hi = (
            _safe_ci(boot_diff[:, j], alpha) if pd.notna(diff) else (np.nan, np.nan)
        )
        rows.append(
            {
                "incoming": incoming,
                "locked_guess": locked_rules.loc[incoming, "locked_guess"],
                "habit_n": hn,
                "habit_win_rate": h_rate,
                "break_n": bn,
                "break_win_rate": b_rate,
                "break_minus_habit": diff,
                "diff_ci_low": diff_lo,
                "diff_ci_high": diff_hi,
            }
        )

    payoff = pd.DataFrame(rows).set_index("incoming")
    overall_win_rate = float(valid["player_won"].mean()) if len(valid) > 0 else np.nan
    return payoff, {"outcome_coverage": coverage, "overall_win_rate": overall_win_rate}


def _choose_label_offsets(xs, ys, x_min, x_max, y_min, y_max):
    """Pick non-overlapping (dx, dy) label offsets inside the plot frame."""
    candidates = [
        (14, 18),
        (-14, 18),
        (14, -20),
        (-14, -20),
        (0, 24),
        (0, -26),
        (22, 0),
        (-22, 0),
        (20, 28),
        (-20, 28),
        (20, -30),
        (-20, -30),
    ]
    x_span = max(x_max - x_min, 1e-6)
    y_span = max(y_max - y_min, 1e-6)
    box_w, box_h = 0.25, 0.11
    pt_pad_x, pt_pad_y = 0.045, 0.055

    def _rect_overlap(a, b):
        ox = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
        oy = max(0.0, min(a[3], b[3]) - max(a[2], b[2]))
        return ox * oy

    placed_rects = []
    offsets = [None] * len(xs)
    order = np.argsort(-np.asarray(ys))
    pts_norm = np.column_stack(
        [(np.asarray(xs) - x_min) / x_span, (np.asarray(ys) - y_min) / y_span]
    )

    for idx in order:
        best_score = -1e9
        best_cand = candidates[0]
        best_rect = None
        for dx, dy in candidates:
            anchor = pts_norm[idx] + np.array([dx / 250.0, dy / 175.0])
            x0 = anchor[0] if dx > 0 else anchor[0] - box_w if dx < 0 else anchor[0] - box_w / 2
            y0 = anchor[1] if dy > 0 else anchor[1] - box_h if dy < 0 else anchor[1] - box_h / 2
            rect = (x0, x0 + box_w, y0, y0 + box_h)

            overflow = (
                max(0.0, 0.02 - rect[0])
                + max(0.0, rect[1] - 0.98)
                + max(0.0, 0.03 - rect[2])
                + max(0.0, rect[3] - 0.87)
            )
            # Penalize overlap with already-placed label boxes and all bubbles.
            box_overlap = sum(_rect_overlap(rect, r) for r in placed_rects)
            pt_overlap = sum(
                _rect_overlap(
                    rect,
                    (p[0] - pt_pad_x, p[0] + pt_pad_x, p[1] - pt_pad_y, p[1] + pt_pad_y),
                )
                for p in pts_norm
            )
            # Also avoid the bottom-right legend box.
            legend_overlap = _rect_overlap(rect, (0.50, 0.99, 0.0, 0.19))
            dist_pen = np.hypot(dx / 250.0, dy / 175.0) * 0.08
            score = -120.0 * box_overlap - 90.0 * pt_overlap - 80.0 * legend_overlap - 15.0 * overflow - dist_pen
            if score > best_score:
                best_score = score
                best_cand = (dx, dy)
                best_rect = rect
        offsets[idx] = best_cand
        placed_rects.append(best_rect)
    return offsets


def plot_scouting_priority_map(report, player_name=PLAYER):
    """Chart 1: Practice Priority Map (hit rate vs. margin, sized by volume)."""
    df = report.dropna(subset=["hit_rate", "margin"]).copy()
    fig, ax = plt.subplots(figsize=(10.8, 6.4), dpi=130)
    fig.patch.set_facecolor("white")

    x_vals = df["margin"].to_numpy() * 100
    y_vals = df["hit_rate"].to_numpy() * 100
    x_lo = (df["margin"] - df["margin_ci_low"]).to_numpy() * 100
    x_hi = (df["margin_ci_high"] - df["margin"]).to_numpy() * 100
    y_lo = (df["hit_rate"] - df["hit_ci_low"]).to_numpy() * 100
    y_hi = (df["hit_ci_high"] - df["hit_rate"]).to_numpy() * 100

    x_min = min(-10.0, float(np.nanmin(df["margin_ci_low"] * 100)) - 4.0)
    x_max = max(42.0, float(np.nanmax(df["margin_ci_high"] * 100)) + 10.0)
    y_min = max(15.0, float(np.nanmin(df["hit_ci_low"] * 100)) - 6.0)
    y_max = min(75.0, float(np.nanmax(df["hit_ci_high"] * 100)) + 12.0)

    # Shaded coaching zones: left = balanced choice, upper-right = readable habit.
    ax.axvspan(x_min, 5.0, color="#ECFDF5", alpha=0.55, zorder=0)
    ax.fill_between(
        [15.0, x_max], 40.0, y_max, color="#FEF2F2", alpha=0.6, zorder=0
    )
    ax.axvline(0.0, color=PALETTE["neutral"], linestyle="--", linewidth=1.1, zorder=1)
    ax.axhline(33.3, color=PALETTE["neutral"], linestyle=":", linewidth=1.0, zorder=1)

    ax.text(
        x_min + 0.9,
        y_max - 1.4,
        "COIN-FLIP ZONE\n(Balanced — keep choice)",
        fontsize=8.5,
        color="#065F46",
        fontweight="bold",
        va="top",
    )
    ax.text(
        x_max - 0.9,
        y_max - 1.4,
        "SIT-ON-IT ZONE\n(Drill redirects & disguise)",
        fontsize=8.5,
        color="#991B1B",
        fontweight="bold",
        va="top",
        ha="right",
    )
    ax.text(
        x_max - 0.9,
        33.8,
        "33.3% (1-in-3 aim baseline)",
        fontsize=8,
        color=PALETTE["neutral"],
        ha="right",
        va="bottom",
    )

    max_n = max(float(df["test_n"].max()), 1.0)
    sizes = 160 + 520 * (df["test_n"].to_numpy() / max_n)
    colors = [
        PALETTE["fh_primary"] if inc.startswith("f") else PALETTE["bh_primary"]
        for inc in df.index
    ]

    for i, incoming in enumerate(df.index):
        ax.errorbar(
            x_vals[i],
            y_vals[i],
            xerr=[[x_lo[i]], [x_hi[i]]],
            yerr=[[y_lo[i]], [y_hi[i]]],
            fmt="none",
            ecolor=colors[i],
            elinewidth=1.4,
            capsize=3,
            alpha=0.55,
            zorder=2,
        )
    ax.scatter(
        x_vals,
        y_vals,
        s=sizes,
        c=colors,
        edgecolors="white",
        linewidths=1.6,
        alpha=0.92,
        zorder=3,
    )

    offsets = _choose_label_offsets(x_vals, y_vals, x_min, x_max, y_min, y_max)
    for i, (incoming, row) in enumerate(df.iterrows()):
        dx, dy = offsets[i]
        ha = "left" if dx > 0 else "right" if dx < 0 else "center"
        va = "bottom" if dy > 0 else "top" if dy < 0 else "center"
        label = (
            f"{incoming} ({SHOT_NAMES[incoming]}) → {row['locked_guess']}\n"
            f"{row['hit_rate']:.1%} · margin {row['margin']:+.1%} · n={int(row['test_n']):,}"
        )
        ax.annotate(
            label,
            xy=(x_vals[i], y_vals[i]),
            xytext=(dx, dy),
            textcoords="offset points",
            fontsize=8.3,
            fontweight="bold",
            color=PALETTE["slate"],
            ha=ha,
            va=va,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                edgecolor=colors[i],
                alpha=0.9,
                linewidth=0.9,
            ),
            arrowprops=dict(arrowstyle="-", color=colors[i], lw=0.9, alpha=0.7),
            zorder=4,
        )

    legend_handles = [
        Line2D(
            [0], [0], marker="o", color="w", markerfacecolor=PALETTE["fh_primary"],
            markersize=9, label="Incoming Opponent Forehand (f1–f3)"
        ),
        Line2D(
            [0], [0], marker="o", color="w", markerfacecolor=PALETTE["bh_primary"],
            markersize=9, label="Incoming Opponent Backhand (b1–b3)"
        ),
        Line2D(
            [0], [0], color=PALETTE["neutral"], lw=1.4,
            label="Crosshairs = 95% match-bootstrap CI · Size = match volume"
        ),
    ]
    ax.legend(
        handles=legend_handles,
        loc="lower right",
        frameon=True,
        facecolor="white",
        edgecolor=PALETTE["grid"],
        fontsize=8.5,
    )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel(
        "Habit Margin Over #2 Option (Held-Out Hit Rate − Runner-Up Rate, percentage points)",
        fontsize=10,
        labelpad=8,
    )
    ax.set_ylabel(
        "Held-Out Hit Rate of Locked #1 Habit (%)", fontsize=10, labelpad=8
    )
    ax.set_title(
        f"{player_name}: Practice Priority Map — Where Opponents Can Sit on Your Shot",
        fontsize=12.5,
        fontweight="bold",
        pad=14,
        loc="left",
    )
    ax.grid(True, linestyle="--", alpha=0.45, color=PALETTE["grid"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    return fig, ax


def plot_shot_response_profiles(
    train_probs, eval_df, report, labels=LABELS, player_name=PLAYER
):
    """Chart 2: 2x3 Court-Third Shot Map with wing/aim split and training drift."""
    test_matrix = build_response_matrix(eval_df)
    test_probs = test_matrix.div(test_matrix.sum(axis=1).replace(0, np.nan), axis=0)

    max_prob = float(
        np.nanmax(
            [
                test_probs.to_numpy(dtype=float).max(),
                train_probs.to_numpy(dtype=float).max(),
            ]
        )
    )
    y_top = min(1.0, max(0.62, max_prob + 0.13))

    fig, axes = plt.subplots(2, 3, figsize=(14.0, 7.6), dpi=130, sharey=True)
    fig.patch.set_facecolor("white")
    x = np.arange(len(labels))

    for idx, incoming in enumerate(labels):
        ax = axes[idx // 3, idx % 3]
        row = report.loc[incoming]
        guess = row["locked_guess"]
        runner = row["runner_up"]

        t_probs = test_probs.loc[incoming].reindex(labels).to_numpy(dtype=float)
        tr_probs = train_probs.loc[incoming].reindex(labels).to_numpy(dtype=float)

        bar_colors = []
        for lbl in labels:
            is_fh = lbl.startswith("f")
            if lbl == guess:
                bar_colors.append(
                    PALETTE["fh_primary"] if is_fh else PALETTE["bh_primary"]
                )
            elif lbl == runner:
                bar_colors.append(
                    PALETTE["fh_secondary"] if is_fh else PALETTE["bh_secondary"]
                )
            else:
                bar_colors.append(
                    PALETTE["fh_muted"] if is_fh else PALETTE["bh_muted"]
                )

        ax.bar(x, np.nan_to_num(t_probs, nan=0.0), width=0.66, color=bar_colors, zorder=2)
        ax.scatter(
            x,
            tr_probs,
            marker="D",
            s=34,
            facecolor="white",
            edgecolor=PALETTE["slate"],
            linewidth=1.3,
            clip_on=False,
            zorder=4,
        )
        ax.axvline(2.5, color=PALETTE["grid"], linestyle="-", linewidth=1.1, zorder=1)

        for j, lbl in enumerate(labels):
            val = t_probs[j]
            if np.isnan(val):
                continue
            if lbl in (guess, runner) or val >= 0.08:
                top_y = max(val, tr_probs[j] if not np.isnan(tr_probs[j]) else 0.0)
                tag = "#1 " if lbl == guess else "#2 " if lbl == runner else ""
                ax.text(
                    j,
                    top_y + 0.016,
                    f"{tag}{val:.0%}",
                    ha="center",
                    va="bottom",
                    fontsize=8.2 if lbl == guess else 7.6,
                    fontweight="bold" if lbl == guess else "normal",
                    color=PALETTE["slate"],
                )

        wing_lbl = "FH" if row["locked_wing"] == "f" else "BH"
        subtitle = (
            f"{wing_lbl} {row['wing_rate']:.0%} · Aim→{guess}: {row['dir_given_wing']:.0%} "
            f"· Drift {row['drift']:+.1%}"
            if pd.notna(row["hit_rate"])
            else "No held-out transitions"
        )
        ax.set_title(
            f"Opp {incoming} ({SHOT_NAMES[incoming]}) · n={int(row['test_n']):,}\n{subtitle}",
            fontsize=9.5,
            fontweight="bold",
            pad=8,
            color=PALETTE["slate"],
        )
        ax.set_xticks(x)
        ax.set_xticklabels(
            [f"{lbl}\n{SHOT_NAMES[lbl].split()[1]}" for lbl in labels], fontsize=8.2
        )
        ax.set_ylim(0.0, y_top)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
        ax.grid(axis="y", linestyle="--", alpha=0.45, color=PALETTE["grid"])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if idx % 3 == 0:
            ax.set_ylabel("Held-Out Reply Share", fontsize=9.5)

    legend_handles = [
        Patch(facecolor=PALETTE["fh_primary"], label="#1 Habit (FH)"),
        Patch(facecolor=PALETTE["fh_secondary"], label="#2 Runner-Up (FH)"),
        Patch(facecolor=PALETTE["bh_primary"], label="#1 Habit (BH)"),
        Patch(facecolor=PALETTE["bh_secondary"], label="#2 Runner-Up (BH)"),
        Line2D(
            [0], [0], marker="D", color="w", markerfacecolor="white",
            markeredgecolor=PALETTE["slate"], markeredgewidth=1.3,
            markersize=6.5, label="Pre-Cutoff Training Share"
        ),
    ]
    fig.suptitle(
        f"{player_name}: Court-Third Reply Map & Training-to-Test Drift",
        fontsize=13,
        fontweight="bold",
        y=0.98,
    )
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.935),
        ncol=5,
        frameon=False,
        fontsize=8.8,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    return fig, axes


def plot_habit_payoff(payoff_df, payoff_meta, report, player_name=PLAYER):
    """Chart 3: Does the predictable habit win or lose points vs. breaking pattern?"""
    order = (
        report["margin"]
        .sort_values(ascending=True, na_position="first")
        .index.tolist()
    )
    df = payoff_df.loc[order].copy()
    y = np.arange(len(df))

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(13.2, 5.6), dpi=130, sharey=True,
        gridspec_kw={"width_ratios": [1.15, 1.0]},
    )
    fig.patch.set_facecolor("white")

    # Left panel: paired dumbbell of Habit vs. Pattern-Break point win rate.
    ax1.axvline(50.0, color=PALETTE["neutral"], linestyle="--", linewidth=1.0, zorder=1)
    if pd.notna(payoff_meta["overall_win_rate"]):
        ovr = payoff_meta["overall_win_rate"] * 100
        ax1.axvline(ovr, color="#9CA3AF", linestyle=":", linewidth=1.1, zorder=1)
        ax1.text(
            ovr, len(df) - 0.32, f"Overall {ovr:.1f}%",
            fontsize=7.8, color="#4B5563", ha="center", va="top"
        )

    all_rates = np.concatenate(
        [df["habit_win_rate"].to_numpy() * 100, df["break_win_rate"].to_numpy() * 100]
    )
    valid_rates = all_rates[np.isfinite(all_rates)]
    r_min = max(25.0, float(valid_rates.min()) - 8.5) if valid_rates.size else 35.0
    r_max = min(85.0, float(valid_rates.max()) + 8.5) if valid_rates.size else 70.0

    for i, (incoming, row) in enumerate(df.iterrows()):
        h_pct = row["habit_win_rate"] * 100
        b_pct = row["break_win_rate"] * 100
        if np.isnan(h_pct) or np.isnan(b_pct):
            continue
        seg_color = PALETTE["break_win"] if b_pct >= h_pct else PALETTE["habit_win"]
        ax1.plot([h_pct, b_pct], [i, i], color=seg_color, linewidth=2.4, alpha=0.8, zorder=2)
        ax1.scatter(
            h_pct, i, s=85, color=PALETTE["habit_win"], edgecolors="white",
            linewidth=1.2, zorder=3
        )
        ax1.scatter(
            b_pct, i, s=85, marker="D", color=PALETTE["break_win"], edgecolors="white",
            linewidth=1.2, zorder=3
        )
        # Place lower label left and higher label right to prevent collisions.
        if h_pct <= b_pct:
            ax1.text(h_pct - 1.1, i, f"{h_pct:.1f}%", ha="right", va="center", fontsize=8.3, color=PALETTE["habit_win"], fontweight="bold")
            ax1.text(b_pct + 1.1, i, f"{b_pct:.1f}%", ha="left", va="center", fontsize=8.3, color=PALETTE["break_win"], fontweight="bold")
        else:
            ax1.text(b_pct - 1.1, i, f"{b_pct:.1f}%", ha="right", va="center", fontsize=8.3, color=PALETTE["break_win"], fontweight="bold")
            ax1.text(h_pct + 1.1, i, f"{h_pct:.1f}%", ha="left", va="center", fontsize=8.3, color=PALETTE["habit_win"], fontweight="bold")

    y_labels = [
        f"{inc} ({SHOT_NAMES[inc]})\nHabit: {df.loc[inc, 'locked_guess']}"
        for inc in df.index
    ]
    ax1.set_yticks(y)
    ax1.set_yticklabels(y_labels, fontsize=8.8)
    ax1.set_ylim(-0.6, len(df) - 0.15)
    ax1.set_xlim(r_min, r_max)
    ax1.set_xlabel("Held-Out Point Win Rate (%)", fontsize=9.8)
    ax1.set_title("Point Win Rate: Locked Habit vs. Pattern Break", fontsize=10.5, fontweight="bold", pad=12)
    ax1.grid(axis="x", linestyle="--", alpha=0.45, color=PALETTE["grid"])
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    ax1.legend(
        handles=[
            Line2D([0], [0], marker="o", color="w", markerfacecolor=PALETTE["habit_win"], markersize=8, label="Locked #1 Habit"),
            Line2D([0], [0], marker="D", color="w", markerfacecolor=PALETTE["break_win"], markersize=7.5, label="Pattern Break (Other Reply)"),
        ],
        loc="lower right",
        frameon=True,
        facecolor="white",
        edgecolor=PALETTE["grid"],
        fontsize=8.3,
    )

    # Right panel: forest plot of (Pattern Break - Habit) win-rate delta with 95% match CI.
    ax2.axvline(0.0, color=PALETTE["slate"], linestyle="--", linewidth=1.1, zorder=1)
    diffs = df["break_minus_habit"].to_numpy() * 100
    ci_lo = df["diff_ci_low"].to_numpy() * 100
    ci_hi = df["diff_ci_high"].to_numpy() * 100
    finite_lo = ci_lo[np.isfinite(ci_lo)]
    finite_hi = ci_hi[np.isfinite(ci_hi)]
    d_min = min(-8.0, float(finite_lo.min()) - 2.5) if finite_lo.size else -12.0
    d_max_bar = max(8.0, float(finite_hi.max()) + 2.5) if finite_hi.size else 12.0
    d_max = d_max_bar + 0.56 * (d_max_bar - d_min)

    for i, (incoming, row) in enumerate(df.iterrows()):
        if np.isnan(diffs[i]):
            continue
        if ci_lo[i] > 0:
            c = PALETTE["break_win"]
        elif ci_hi[i] < 0:
            c = PALETTE["habit_win"]
        else:
            c = PALETTE["neutral"]
        ax2.errorbar(
            diffs[i],
            i,
            xerr=[[diffs[i] - ci_lo[i]], [ci_hi[i] - diffs[i]]],
            fmt="o",
            color=c,
            markersize=7,
            elinewidth=2.0,
            capsize=3.5,
            zorder=3,
        )
        ax2.text(
            d_max_bar + 0.4,
            i,
            f"{diffs[i]:+.1f} pts ({int(row['habit_n'])} vs {int(row['break_n'])})",
            va="center",
            ha="left",
            fontsize=8.1,
            color=c,
            fontweight="bold" if c != PALETTE["neutral"] else "normal",
        )

    ax2.set_xlim(d_min, d_max)
    ticks = [t for t in ax2.get_xticks() if d_min <= t <= d_max_bar]
    ax2.set_xticks(ticks)
    ax2.set_xlabel("Pattern Break − Habit Win-Rate Gap (pts, 95% match CI)", fontsize=9.8)
    ax2.set_title("Net Payoff of Breaking Pattern (Sorted by Habit Margin)", fontsize=10.5, fontweight="bold", pad=12)
    ax2.text(d_min + 0.6, len(df) - 0.32, "← Habit wins more", fontsize=8, color=PALETTE["habit_win"], va="top")
    ax2.text(d_max_bar - 0.6, len(df) - 0.32, "Pattern break wins more →", fontsize=8, color=PALETTE["break_win"], ha="right", va="top")
    ax2.grid(axis="x", linestyle="--", alpha=0.45, color=PALETTE["grid"])
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    fig.suptitle(
        f"{player_name}: Habit Payoff — Should You Keep the Habit or Break Pattern?",
        fontsize=13,
        fontweight="bold",
        y=0.97,
    )
    fig.text(
        0.5,
        0.012,
        "Caveat: Point-win rates do not control for incoming ball depth/speed (excluded by parser); "
        "short balls are easier to redirect and win, so confirm context on video before forcing pattern breaks. "
        f"(Outcome coverage: {payoff_meta['outcome_coverage']:.1%})",
        ha="center",
        fontsize=8.2,
        color="#4B5563",
        style="italic",
    )
    fig.tight_layout(rect=[0, 0.045, 1, 0.92])
    return fig, (ax1, ax2)


def _check_visualizations():
    synth_points = pd.DataFrame(
        [
            {"match_id": "m1", "Pt": 1, "PtWinner": 1, "player_side": 1},
            {"match_id": "m1", "Pt": 2, "PtWinner": 2, "player_side": 1},
            # Duplicate consistent row must collapse cleanly without multiplying rows.
            {"match_id": "m1", "Pt": 2, "PtWinner": 2, "player_side": 1},
            # Conflicting duplicate must become NaN rather than arbitrarily picking one.
            {"match_id": "m1", "Pt": 3, "PtWinner": 1, "player_side": 1},
            {"match_id": "m1", "Pt": 3, "PtWinner": 2, "player_side": 1},
        ]
    )
    synth_eval = pd.DataFrame(
        [
            {"match_id": "m1", "point_number": 1, "incoming": "f3", "actual_response": "b3"},
            {"match_id": "m1", "point_number": 2, "incoming": "f3", "actual_response": "b2"},
            {"match_id": "m1", "point_number": 3, "incoming": "f3", "actual_response": "b3"},
        ]
    )
    attached = attach_point_outcomes(synth_eval, synth_points)
    assert len(attached) == 3
    assert bool(attached.loc[0, "player_won"]) is True
    assert bool(attached.loc[1, "player_won"]) is False
    assert pd.isna(attached.loc[2, "player_won"])

    locked = pd.DataFrame(
        [{"incoming": lbl, "locked_guess": "b3" if lbl == "f3" else "f1"} for lbl in LABELS]
    ).set_index("incoming")
    payoff, meta = compute_habit_payoff(attached, locked, n_boot=100, seed=0)
    assert np.isclose(payoff.loc["f3", "habit_win_rate"], 1.0)
    assert np.isclose(payoff.loc["f3", "break_win_rate"], 0.0)
    assert np.isclose(payoff.loc["f3", "break_minus_habit"], -1.0)
    assert np.isclose(meta["outcome_coverage"], 2 / 3)


_check_visualizations()
print("visualization and payoff checks passed")

locked_rules_viz = rank_training_responses(train_matrix, labels=LABELS)
eval_with_outcomes = attach_point_outcomes(evaluation, player_points)
habit_payoff, habit_payoff_meta = compute_habit_payoff(
    eval_with_outcomes, locked_rules_viz, labels=LABELS, n_boot=2000, seed=0
)

fig1, _ = plot_scouting_priority_map(predictability_report, player_name=PLAYER)
fig2, _ = plot_shot_response_profiles(
    train_probs, evaluation, predictability_report, labels=LABELS, player_name=PLAYER
)
fig3, _ = plot_habit_payoff(
    habit_payoff, habit_payoff_meta, predictability_report, player_name=PLAYER
)
plt.show()